# 06 — Full Evaluation

Loads best fusion model checkpoint, runs full benchmark evaluation:
- Optimal threshold selection (maximize sensitivity-weighted F1)
- All classification + clinical metrics with bootstrap 95% CI
- Modality ablation study
- Baseline comparison (LR, RF, GBM, XGBoost on HPO features)
- Generates `reports/benchmark_report.md`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/earlyMind'):
    !git clone https://github.com/Rickykapoor/earlyMind.git /content/earlyMind
%cd /content/earlyMind
!git pull origin main

In [ ]:
!pip install -q mne nibabel nilearn openneuro-py torch torchvision \
  streamlit plotly scikit-learn "dvc[gdrive]" pytorch-tabnet \
  xgboost catboost tqdm pyyaml

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!dvc remote add -d gdrive gdrive://19DlpHZ5QCBcFKvfKHprIUhUeVg5bDxU4 --force
!dvc pull
print('Datasets ready.')

In [ ]:
import sys
sys.path.insert(0, '/content/earlyMind')

import torch
import numpy as np
from src.config import cfg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg.paths.makedirs()

ckpt_path = cfg.paths.checkpoints / 'fusion_model.pt'
if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint missing: {ckpt_path}. Run 05_fusion_train first.')
print(f'Checkpoint found: {ckpt_path}')

In [ ]:
# Load model
from src.models.fusion_model import build_fusion_model

n_hpo = cfg.model.hpo_n_features
model = build_fusion_model(n_hpo=n_hpo)
model.load_state_dict(torch.load(str(ckpt_path), map_location='cpu'))
model = model.to(device)
model.eval()
print('Model loaded.')

In [ ]:
# Build DataLoaders
from src.data.fusion_dataset import build_dataloaders

_, val_ldr, test_ldr = build_dataloaders(
    eeg_processed_dir  = cfg.paths.eeg_processed,
    mri_processed_dir  = cfg.paths.mri_processed,
    hpo_processed_dir  = cfg.paths.hpo_processed,
    eeg_raw_dir        = cfg.paths.eeg_raw,
    batch_size         = 32,
    modality_dropout_p = 0.0,    # no dropout at eval
)
print(f'Val batches: {len(val_ldr)}, Test batches: {len(test_ldr)}')

In [ ]:
# Run inference
from src.training.evaluate import run_inference, find_optimal_threshold, compute_classification_metrics, bootstrap_auc_ci

val_res  = run_inference(model, val_ldr,  device)
test_res = run_inference(model, test_ldr, device)

thr = find_optimal_threshold(val_res['probs'], val_res['labels'])
print(f'Optimal threshold: {thr:.3f}')

metrics = compute_classification_metrics(
    test_res['probs'], test_res['labels'],
    pred_dq=test_res['pred_dq'], true_dq=test_res['true_dq'],
    threshold=thr
)
auc_lo, auc_hi = bootstrap_auc_ci(test_res['probs'], test_res['labels'])

print('\n=== Test Set Metrics ===')
for k, v in metrics.items():
    print(f'  {k:15s}: {v}')
print(f'  AUC 95% CI: [{auc_lo:.3f}, {auc_hi:.3f}]')

In [ ]:
# ROC curve
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc as auc_fn

fpr, tpr, _ = roc_curve(test_res['labels'], test_res['probs'])
roc_auc = auc_fn(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'EarlyMind AUC = {roc_auc:.3f}')
plt.plot([0,1],[0,1],'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — EarlyMind')
plt.legend()
plt.tight_layout()
plt.savefig(str(cfg.paths.reports) + '/roc_curve.png', dpi=100)
plt.show()

In [ ]:
# Ablation study
from src.training.evaluate import ablation_study

ablation_df = ablation_study(model, test_ldr, device)
print('\n=== Ablation Study ===')
display(ablation_df)

In [ ]:
# Baseline comparison on HPO features
from src.training.evaluate import run_baselines

X = np.load(str(cfg.paths.hpo_processed / 'hpo_matrix.npy'))
y = np.load(str(cfg.paths.hpo_processed / 'hpo_labels.npy'))

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

baseline_df = run_baselines(X_tr, y_tr, X_te, y_te)
print('\n=== Baselines ===')
display(baseline_df)

In [ ]:
# Generate full benchmark report
from src.training.evaluate import generate_benchmark_report

report_path = generate_benchmark_report(
    model=model,
    test_loader=test_ldr,
    val_loader=val_ldr,
    device=device,
    X_train_hpo=X_tr, y_train_hpo=y_tr,
    X_test_hpo=X_te,  y_test_hpo=y_te,
    report_dir=str(cfg.paths.reports),
)

with open(report_path) as f:
    print(f.read()[:2000])  # Preview first 2000 chars

In [ ]:
!dvc push
!git add checkpoints/ reports/ datasets/processed/
!git commit -m 'colab: 06_evaluate completed'
!git push origin main